# VESTIGIUM: MODIS–VIIRS S-NPP Harmonization

**NASA Space Apps Challenge 2026 · Team Ga Tidur · Indonesia**

This notebook reproduces the core scientific questions, conditional calibration methods, temporal evaluation, and limitations of a 2000–2025 active-fire detection calendar. Everything is based on original NASA FIRMS records that were previously filtered to `type=0` and aggregated in **25 km EPSG:6933 grid cells per UTC month and sensor**. The inputs are *satellite detections*, **not confirmed fire events, burned area, or proof of an ignition cause**.

## Reproducibility

Run `python src/pipelines/harmonize.py --input data/processed/geospatial/monthly_equal_area_25km.csv --output data/processed/harmonization` first, from repository root. Inputs and figures live in `data/processed/` and are not checked into Git. The `source/` folder contains the upstream advanced-geospatial CSV; `outputs/` contains the results created by `harmonize.py`. Code in this notebook can train and evaluate models again.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'README.md').exists() and (p/'data').is_dir()), None)
assert ROOT is not None, 'Open the notebook from inside the VESTIGIUM repository'
sys.path.insert(0, str(ROOT/'src'/'pipelines'))
from harmonize import (read_input, make_training_frame, fitting_splits, calc_scales,
                       predict_baselines, fit_ml, metric, EXPECTED, FEATURES)
root = ROOT
source = root / 'data' / 'processed' / 'geospatial' / 'monthly_equal_area_25km.csv'
out = root / 'data' / 'processed' / 'harmonization'
assert source.exists(), f'Missing source: {source}'
print('Source:', source, '\nOutputs:', out)

## 1. Inspect the source before calibration

Each row is a sensor/grid/month summary, *not* an individual pixel. The S-NPP reference only begins in 2012; NOAA-20 is kept separate and is **not added** to S-NPP in the calibration target.

In [ ]:
d = read_input(source)
print('Grid–month–sensor rows:', f'{len(d):,}')
display(d.head(4))
display(d.groupby('sensor').agg(grid_month_rows=('detection_count','size'),
                                satellite_pixel_detections=('detection_count','sum')))
assert d.groupby('sensor').detection_count.sum().to_dict() == EXPECTED

### Why grid-month matching is not pixel matching

The two instruments have different pixel footprints and overpass times. Two detections in one grid-month may refer to different events. A month with no detected VIIRS pixel is **not evidence of no fire**; clear-sky observations and orbit opportunity are not available as denominators in this input.

In [ ]:
base, snpp_coverage = make_training_frame(d)
print('MODIS-positive grid-months:', f'{len(base):,}')
print('Reference months with S-NPP pixel detections:', snpp_coverage.year.min(), '–', snpp_coverage.year.max())
print('Example matched cells:')
display(base.loc[base.year.eq(2015),['year','month','modis_count','snpp_count','modis_sum_frp_mw']].head(6))

## 2. Temporal validation to reduce leakage

Train on **2013–2021** (2012 is a partial launch year), select the method on **2022** validation, and report outcomes on the untouched **2023–2025** test. The test deliberately follows the validation period in calendar time.

In [ ]:
splits = fitting_splits(base)
train, validation, test = [splits[x] for x in ('train','validation','test')]
print({k: {'n_cell_months':len(v),'years':(int(v.year.min()),int(v.year.max()))}
       for k,v in splits.items()})
assert train.year.max() < validation.year.min() < test.year.min()

## 3. Three harmonization candidates plus an unharmonized comparator

1. Raw MODIS count (**comparator only**, not a calibration).
2. Global multiplicative ratio derived from training-period reference detections.
3. Seasonally varying ratio regularized toward the global ratio.
4. HistGradientBoostingRegressor with Poisson loss and MODIS-derived count, FRP, season, and geographic features.

No target/reference observations from the validation or test years are used to fit coefficients.

In [ ]:
global_ratio, seasonal_ratios = calc_scales(train)
print('Training global S-NPP/MODIS ratio:', round(global_ratio,3))
display(pd.DataFrame({'month':list(seasonal_ratios),'ratio':list(seasonal_ratios.values())}))

In [ ]:
model = fit_ml(train)
def all_predictions(frame):
    p = predict_baselines(frame,global_ratio,seasonal_ratios)
    p['ml_poisson_hgbr'] = model.predict(frame[FEATURES].to_numpy())
    return p
pred_val = all_predictions(validation)
pred_test = all_predictions(test)
print('Poisson ML fitted using 2013–2021 only.')

## 4. Model selection on 2022 only

The validation criterion is **minimum RMSE** among the 3 actual harmonization candidates. Raw MODIS remains a diagnostic comparator, since selecting uncalibrated counts would defeat the scientific question. MAE, WAPE, Spearman rank correlation, and volume bias are also reported, and they may disagree with RMSE.

In [ ]:
val_scores = pd.DataFrame({name:metric(validation,p) for name,p in pred_val.items()}).T
selected = min(['global_ratio','seasonal_ratio','ml_poisson_hgbr'], key=lambda x:val_scores.loc[x,'rmse'])
print('Chosen method:',selected)
display(val_scores[['mae','rmse','wape','volume_bias_pct','spearman_rho']].round(3))
assert selected == json.loads((out/'evaluation.json').read_text())['preferred_method']

## 5. Untouched 2023–2025 test

A lower MAE does not imply better RMSE or unbiased volume. Test metrics apply only where **MODIS > 0**, while the full S-NPP national series includes **additional VIIRS-only cells**. These targets are not interchangeable.

In [ ]:
test_scores = pd.DataFrame({name:metric(test,p) for name,p in pred_test.items()}).T
display(test_scores[['n_cells','mae','rmse','wape','volume_bias_pct','spearman_rho']].round(3))
print('Minimum test MAE:',test_scores.mae.idxmin())
print('Minimum test RMSE:',test_scores.rmse.idxmin())

In [ ]:
display(Image(filename=str(out/'figures'/'03_temporal_holdout_2023_2025.png')))

## 6. MODIS-only history has an unavoidable coverage blind spot

S-NPP detects many pixels in grid-months where MODIS reports no presumed vegetation fire. A model that *only* calibrates MODIS-positive cells cannot infer these missing spatial locations from the MODIS active-fire dataset alone.

In [ ]:
coverage = pd.read_csv(out/'overlap_coverage_diagnostics.csv')
coverage['SNPP_outside_MODIS_share_pct'] = 100*(1-coverage['snpp_pixels_in_modis_positive_share'])
display(coverage[['year','snpp_total','snpp_in_modis_positive_cells',
                  'SNPP_outside_MODIS_share_pct']].round(2))
display(Image(filename=str(out/'figures'/'05_modis_blind_spot_diagnostic.png')))

## 7. Spatial and temporal sensitivity

The following figures document error by approximate longitude region, not province. The method is *not* equally accurate everywhere. This experiment does **not** establish biome-specific or cloud-corrected calibration.

In [ ]:
display(pd.read_csv(out/'test_regional_diagnostics.csv').round(2))
display(Image(filename=str(out/'figures'/'04_regional_bias.png')))

## 8. Deliverable: historical monthly data with explicit provenance

`modis_reference_equivalent_proxy` is a continuous, MODIS-conditioned **modeled detection proxy** over 2000–2025. The separate `calendar_display_estimate` splices modeled MODIS (2000–2011) and **direct** S-NPP observations (2012+); **that hybrid has a structural measurement break**, which must never be interpreted as a true increase/decrease in fire activity. The record for year 2000 is November–December only.

In [ ]:
calendar = pd.read_csv(out/'harmonized_calendar_monthly.csv')
annual = pd.read_csv(out/'harmonized_calendar_annual.csv')
display(annual.head(4))
display(annual.tail(4))
assert annual.loc[annual.year.eq(2000),'incomplete_year'].iloc[0]
assert calendar.loc[calendar.year.eq(2001),'VIIRS_SNPP'].isna().all()
assert (calendar['cross_2012_discontinuity_warning']).all()

In [ ]:
display(Image(filename=str(out/'figures'/'06_harmonized_monthly_heatmap.png')))

## 9. Integration and limitations

- Monthly **grid cells** can be rendered as heat/grid density overlays; preserve the underlying 25 km cell definition and provenance labels.
- Red clickable **individual** hotspot markers must come from the **separate per-detection SQLite database**, not from fractional calibrated grid counts.
- The data do **not** label individual ignition causes. Do not train a binary arson-vs-natural-fire classifier from these records alone.
- `type=0` means *presumed vegetation fire*. Cross-sensor confidence values have incompatible encodings and are deliberately not used as if equivalent.
- The selected ML model had smaller test MAE but larger test RMSE than a statistical baseline. Avoid blanket claims of superior accuracy.

**This is an initial calibrated proxy**, not a completed science-grade active fire product. Critical future work includes satellite observation-coverage correction, near-time co-observation matching, independently validated event data, biome stratification, blocked spatial testing, and calibrated uncertainty intervals.

See `VESTIGIUM_HARMONIZATION_REPORT.md`, `DATA_CONTRACT.md`, and `outputs/evaluation.json`.